# 4D minimal-cell simulation + CellGNN learned surrogate

Run on Colab. End-to-end:
1. clone the repo on Colab + verify torch
2. run our own `cell_physics` engine (Brownian dynamics + soft LJ + FENE chain + complex restraints + sphere wall + proximity reactions) to produce a 4D trajectory of a syn3A-flavoured cell
3. load the trajectory + report shapes + plot count traces
4. visualize one frame in 3D
5. build heterogeneous dynamic edges from a frame
6. forward-pass `CellGNN` on the edges + check E(3) equivariance
7. assemble `(t, t+dt)` training pairs
8. tiny next-frame-displacement training loop
9. roll the trained model forward at 1 μs cadence
10. compare rollout to physics ground truth

**What this is:** a real 4D simulation (the cell_physics part) plus a learned dynamics *surrogate* trained to imitate it (the CellGNN part). The surrogate distils observed dynamics into a fast inference path — same role GraphCast plays for weather, NequIP plays for MD.

**What this is NOT:** a complete syn3A simulator. The physics engine deliberately excludes ribosome biogenesis (Earnest, Lai, Chen 2015 — 145 assembly intermediates), DNA replication, membrane growth, and cell division. Each of those needs its own physics module; we do not pretend an edge type fixes them. v1 simulates diffusion + reactions + a static chromosome polymer + complexes inside a fixed sphere — a real partial-4D cell sim, scoped honestly.

No Google Drive mount, no Zenodo download — uses what's already in the repo.

## 1. Setup

In [ ]:
import os, sys, subprocess
REPO = '/content/cell'
if not os.path.isdir(REPO):
    !git clone https://github.com/Nikku03/cell.git {REPO}
%cd {REPO}
!git fetch --quiet && git checkout claude/syn3a-whole-cell-simulator-REjHC && git pull --quiet --ff-only
sys.path.insert(0, REPO)
sys.path.insert(0, REPO + '/cell_sim')

import torch, numpy as np, pandas as pd, matplotlib.pyplot as plt
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(),
      'device', 'cuda' if torch.cuda.is_available() else 'cpu')
print('numpy', np.__version__)

## 2. Generate a multi-trajectory training dataset

For learned-dynamics training, **multiple short trajectories with different random seeds beat one long trajectory**. The GNN sees diverse initial conditions instead of just temporal correlations from one starting point — same reason ML training uses random batches over different examples instead of one sample replayed.

We run `cell_physics` 10 times in-process (no subprocess overhead) with different seeds. Each run is 200 ns at 1 ns frame cadence → 200 frames per run × 10 runs = **2000 training frames**. Hold out 2 trajectories as validation.

**Wall time estimate:** ~60–90 min on Colab CPU (~6 min per 200 ns run × 10 runs). With Colab Pro + A100, the data gen is still CPU-bound (it's numpy physics) so the GPU only helps the training in cell 9. If you need it faster, drop `N_RUNS` to 6 and `DURATION_NS` to 100 — runs in ~20 min, gives 600 training frames instead of 2000.

In [ ]:
from pathlib import Path
import time
sys.path.insert(0, str(Path(REPO) / 'cell_sim'))
sys.path.insert(0, REPO)

from cell_sim.atom_engine.cell_physics import (
    CellPhysicsConfig, initial_state, run as physics_run,
)
from scripts.run_4d_minimal_cell import (
    species_table, reactions, initial_composition, chromosome_chain,
)

# Configuration
N_RUNS = 10           # 8 train, 2 val
DURATION_NS = 200     # per run
DT_NS = 0.01          # 10 ps integration
SAVE_EVERY = 100      # 1 ns frame cadence
CELL_RADIUS_NM = 120  # tighter sphere for higher reaction density
N_CHAIN_BEADS = 64

species = species_table()
rxns = reactions()
print(f'plan: {N_RUNS} runs × {DURATION_NS} ns = {N_RUNS * DURATION_NS} ns total')
print(f'      {DURATION_NS // (DT_NS * SAVE_EVERY) * N_RUNS:.0f} total frames at '
      f'{DT_NS * SAVE_EVERY} ns cadence')

trajectories = []
t0 = time.time()
for seed in range(N_RUNS):
    config = CellPhysicsConfig(cell_radius_nm=CELL_RADIUS_NM, dt_ns=DT_NS,
                                seed=seed, pair_rebuild_every=20)
    state = initial_state(species, initial_composition(), config,
                          chains=[chromosome_chain(N_CHAIN_BEADS)])
    n_steps = int(DURATION_NS / DT_NS)
    traj = physics_run(state, config, species, reactions=rxns,
                        n_steps=n_steps, save_every=SAVE_EVERY)
    trajectories.append({
        'positions': traj['positions'],
        'species_id': traj['species_id'],
        'counts': traj['counts'],
        't_ns': traj['t_ns'],
        'n_rxn_total': int(traj['n_rxn_total']),
    })
    print(f'  seed {seed:>2d}: {traj["positions"].shape[0]} frames  '
          f'rxn={traj["n_rxn_total"]}  '
          f'wall={time.time()-t0:.0f}s')

# Save concatenated training set as a single npz with index offsets
all_pos = [tr['positions'] for tr in trajectories]
all_sid = [tr['species_id'] for tr in trajectories]
# Frames are (T_i, N_max_i, 3) — N_max varies. Pad to global max for storage.
N_max_global = max(p.shape[1] for p in all_pos)
n_total_frames = sum(p.shape[0] for p in all_pos)
positions_pad = np.full((n_total_frames, N_max_global, 3), np.nan, dtype=np.float32)
species_pad = np.full((n_total_frames, N_max_global), -1, dtype=np.int16)
traj_id = np.zeros(n_total_frames, dtype=np.int32)
frame_offset = 0
for ti, (p, s) in enumerate(zip(all_pos, all_sid)):
    nf, n_p = p.shape[0], p.shape[1]
    positions_pad[frame_offset:frame_offset + nf, :n_p] = p
    species_pad[frame_offset:frame_offset + nf, :n_p] = s
    traj_id[frame_offset:frame_offset + nf] = ti
    frame_offset += nf

DATASET_PATH = Path(REPO) / 'outputs/cell_train_dataset.npz'
np.savez_compressed(
    DATASET_PATH,
    positions=positions_pad,
    species_id=species_pad,
    traj_id=traj_id,
    species=np.array([s.name for s in species]),
    cell_radius_nm=np.float32(CELL_RADIUS_NM),
    n_runs=np.int32(N_RUNS),
)
print(f'\ndone in {time.time()-t0:.0f}s')
print(f'wrote {DATASET_PATH}  ({DATASET_PATH.stat().st_size/1e6:.1f} MB)')
print(f'shape: positions={positions_pad.shape}  traj_id={traj_id.shape}')

## 3. Load training dataset + report

Load the multi-trajectory dataset and prepare a train/val split. Pick 2 of 10 trajectories as held-out validation.

In [ ]:
d = np.load(DATASET_PATH, allow_pickle=True)
POSITIONS = d['positions']        # (Total_frames, N_max, 3)
SPECIES_ID = d['species_id']      # (Total_frames, N_max)
TRAJ_ID = d['traj_id']             # (Total_frames,) which trajectory each frame belongs to
SPECIES_NAMES = list(d['species'])
N_RUNS = int(d['n_runs'])
T_FRAMES = POSITIONS.shape[0]

# Train / val split: last 2 trajectories are validation
N_VAL = 2
N_TRAIN = N_RUNS - N_VAL
TRAIN_MASK = TRAJ_ID < N_TRAIN
VAL_MASK = TRAJ_ID >= N_TRAIN

print(f'positions:       {POSITIONS.shape}')
print(f'trajectories:    {N_RUNS} ({N_TRAIN} train, {N_VAL} val)')
print(f'frames train:    {TRAIN_MASK.sum()}  val: {VAL_MASK.sum()}')
print(f'species:         {len(SPECIES_NAMES)}')

# Plot one trajectory's count traces just so we see what's happening
fig, ax = plt.subplots(figsize=(9, 3.5))
n_per_run = TRAIN_MASK.sum() // N_TRAIN
for i, name in enumerate(SPECIES_NAMES):
    counts = np.array([(SPECIES_ID[t] == i).sum() for t in range(n_per_run)])
    if counts.max() > 0:
        ax.plot(counts, label=name, alpha=0.8)
ax.set_xlabel('frame (1 ns each)'); ax.set_ylabel('copy number')
ax.set_title('species counts — first training trajectory')
ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1.0), fontsize=8)
plt.tight_layout(); plt.show()

## 5. Plot a frame — sanity check the data

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa

f0 = 0
valid = SPECIES_ID[f0] >= 0
p = POSITIONS[f0][valid]
s = SPECIES_ID[f0][valid]
n_show = min(2000, len(p))
idx = np.random.default_rng(0).choice(len(p), size=n_show, replace=False)

fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(111, projection='3d')
sc = ax.scatter(p[idx, 0], p[idx, 1], p[idx, 2], c=s[idx], cmap='tab20', s=4, alpha=0.6)
ax.set_title(f'frame {f0}  ·  {len(p)} particles  ·  showing {n_show}')
ax.set_xlabel('x (nm)'); ax.set_ylabel('y (nm)'); ax.set_zlabel('z (nm)')
fig.colorbar(sc, ax=ax, label='species id', shrink=0.7)
plt.tight_layout(); plt.show()

## 6. Build dynamic heterogeneous edges

Take a single frame, run `build_dynamic_edges`, report the edge-type histogram. This is the moment the cell becomes a graph.

In [ ]:
from cell_sim.atom_engine.cell_gnn import (CellGNN, EdgeType, EdgeFeaturizer,
                                             build_dynamic_edges, N_EDGE_TYPES)

f0 = 0
valid = SPECIES_ID[f0] >= 0
pos_t = torch.tensor(POSITIONS[f0][valid]).float()
sid_t = torch.tensor(SPECIES_ID[f0][valid]).long()    # Embedding wants Long
n_t = pos_t.shape[0]
print(f'frame {f0}: {n_t} particles')

# Subsample if too many (dense pairwise blows up memory)
MAX_N = 3000
if n_t > MAX_N:
    sel = torch.randperm(n_t)[:MAX_N]
    pos_t = pos_t[sel]; sid_t = sid_t[sel]; n_t = MAX_N
    print(f'  subsampled to {n_t} for edge build')

g = build_dynamic_edges(pos_t, sid_t, r_cut_spatial=20.0)
print(f'\nedges: {g["edges"].shape[0]} total')
for et in EdgeType:
    n_e = (g['edge_type'] == int(et)).sum().item()
    if n_e:
        print(f'  {et.name:<11s}  {n_e:>7d}')

import collections
deg = collections.Counter(g['edges'][:, 0].tolist())
deg_arr = np.array(list(deg.values()))
print(f'\nper-particle degree:  median={np.median(deg_arr):.1f}  '
      f'mean={deg_arr.mean():.1f}  max={deg_arr.max()}')

## 7. CellGNN — untrained forward pass + shape + equivariance

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
n_species_vocab = max(int(sid_t.max().item()) + 1, 32)
model = CellGNN(n_species=n_species_vocab, hidden=64, n_rounds=3,
                n_reaction_classes=64, r_cut=20.0,
                n_extra_node_features=4).to(device)
print(f'CellGNN  hidden=64  n_rounds=3  device={device}')
print(f'params: {sum(p.numel() for p in model.parameters()):,}')

extra = torch.zeros((n_t, 4), device=device)
pos_d = pos_t.to(device); sid_d = sid_t.to(device)
edges_d = g['edges'].to(device); r_d = g['r'].to(device)
et_d = g['edge_type'].to(device); thick_d = g['thickness'].to(device)

with torch.no_grad():
    out = model.predict_all(sid_d, extra, edges_d, r_d, et_d, pos_d, thickness=thick_d)
for k, v in out.items():
    print(f'  {k:>22s}  {tuple(v.shape)}')

# Equivariance check on real data
torch.manual_seed(0)
A = torch.randn(3, 3, device=device); Q, _ = torch.linalg.qr(A)
if torch.det(Q) < 0: Q[:, -1] *= -1
pos_rot = pos_d @ Q.T
with torch.no_grad():
    f_orig = model.predict_forces_equivariant(sid_d, extra, edges_d, r_d, et_d, pos_d, thickness=thick_d)
    f_rot = model.predict_forces_equivariant(sid_d, extra, edges_d, r_d, et_d, pos_rot, thickness=thick_d)
err = (f_rot - f_orig @ Q.T).abs().max().item()
print(f'\nE(3) equivariance:  max |f(Rx) - R·f(x)| = {err:.2e}')

## 8. Assemble (t, t+dt) training pairs

We learn next-frame **displacement**: given the current frame's graph + positions, predict `Δposition` to the next frame. This is the simplest learnable signal that doesn't require explicit reaction labels.

Loss: MSE on displacement, restricted to particles present in both frames (matched by frame-major slot ordering for the fallback data; for real LM data, particles need ID-tracking — left as a TODO).

In [ ]:
def make_pair(t):
    """(positions_t, species_t, displacement_t→t+1) for global frame t.
    Only valid when t and t+1 are in the SAME trajectory."""
    if t + 1 >= T_FRAMES or TRAJ_ID[t] != TRAJ_ID[t + 1]:
        return None, None, None
    a_valid = SPECIES_ID[t] >= 0
    b_valid = SPECIES_ID[t + 1] >= 0
    both = a_valid & b_valid
    p_a = POSITIONS[t][both]
    p_b = POSITIONS[t + 1][both]
    s = SPECIES_ID[t][both]
    dp = p_b - p_a
    return p_a, s, dp


# Build train/val frame index lists (only frames where t+1 stays in same traj)
def valid_pair_indices(mask):
    idx = np.where(mask)[0]
    return [t for t in idx if t + 1 < T_FRAMES and TRAJ_ID[t] == TRAJ_ID[t + 1]]

TRAIN_PAIR_IDX = valid_pair_indices(TRAIN_MASK)
VAL_PAIR_IDX   = valid_pair_indices(VAL_MASK)
print(f'training pairs:   {len(TRAIN_PAIR_IDX)}')
print(f'validation pairs: {len(VAL_PAIR_IDX)}')

# Sanity: peek at a pair
p, s, dp = make_pair(TRAIN_PAIR_IDX[0])
print(f'\nsample pair: {p.shape[0]} matched particles')
print(f'displacement: mean |dx|={np.linalg.norm(dp, axis=-1).mean():.2f} nm, '
      f'max={np.linalg.norm(dp, axis=-1).max():.2f} nm')

## 9. Train CellGNN (multi-trajectory, with validation)

30 epochs over 1990 training pairs (8 trajectories) with 400 validation pairs (2 held-out trajectories). Saves the best-val checkpoint.

What good looks like: train and val MSE both decrease smoothly; val plateaus around the same value as train (no overfitting). If val rises while train falls, we're overfitting (smaller hidden / more dropout).

In [ ]:
import time
torch.manual_seed(42)

model = CellGNN(n_species=n_species_vocab, hidden=64, n_rounds=3,
                n_reaction_classes=64, r_cut=20.0,
                n_extra_node_features=4).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-5)
EPOCHS = 30
MAX_N_PAIR = 2000

def step_loss(t_idx):
    """Forward + loss for one training pair. Returns scalar loss tensor."""
    p_a, s, dp = make_pair(int(t_idx))
    if p_a is None or p_a.shape[0] == 0:
        return None
    if p_a.shape[0] > MAX_N_PAIR:
        sel = np.random.default_rng(int(t_idx)).choice(
            p_a.shape[0], size=MAX_N_PAIR, replace=False)
        p_a, s, dp = p_a[sel], s[sel], dp[sel]
    n_p = p_a.shape[0]
    pos = torch.tensor(p_a, device=device).float()
    sid = torch.tensor(s, device=device).long()
    target = torch.tensor(dp, device=device).float()
    ex = torch.zeros((n_p, 4), device=device)
    gg = build_dynamic_edges(pos, sid, r_cut_spatial=20.0)
    pred = model.predict_forces_equivariant(
        sid, ex, gg['edges'].to(device), gg['r'].to(device),
        gg['edge_type'].to(device), pos,
        thickness=gg['thickness'].to(device))
    return (pred - target).pow(2).mean()


train_log, val_log = [], []
best_val = float('inf')
ckpt_dir = Path(REPO) / 'cell_sim/atom_engine/checkpoints'
ckpt_dir.mkdir(parents=True, exist_ok=True)
ckpt_path = ckpt_dir / 'cell_gnn_trained.pt'

rng_t = np.random.default_rng(0)
t0 = time.time()
for ep in range(EPOCHS):
    # ---- train pass ----
    model.train()
    perm = rng_t.permutation(len(TRAIN_PAIR_IDX))
    train_losses = []
    for ix in perm:
        loss = step_loss(TRAIN_PAIR_IDX[ix])
        if loss is None: continue
        opt.zero_grad(); loss.backward(); opt.step()
        train_losses.append(loss.item())
    train_mse = float(np.mean(train_losses)) if train_losses else float('nan')
    train_log.append(train_mse)

    # ---- val pass ----
    model.eval()
    val_losses = []
    with torch.no_grad():
        for t_idx in VAL_PAIR_IDX:
            loss = step_loss(t_idx)
            if loss is not None: val_losses.append(loss.item())
    val_mse = float(np.mean(val_losses)) if val_losses else float('nan')
    val_log.append(val_mse)

    # Best checkpoint
    if val_mse < best_val:
        best_val = val_mse
        torch.save({'state_dict': model.state_dict(),
                    'config': dict(n_species=n_species_vocab, hidden=64,
                                   n_rounds=3, n_reaction_classes=64,
                                   r_cut=20.0, n_extra_node_features=4),
                    'epoch': ep + 1, 'val_mse': best_val},
                   ckpt_path)

    print(f'ep {ep+1:>2d}/{EPOCHS}  train={train_mse:>7.3f}  val={val_mse:>7.3f}  '
          f'(best={best_val:.3f})  wall={time.time()-t0:.0f}s')

print(f'\nbest val MSE: {best_val:.3f} nm²  →  {ckpt_path}')

# Train/val curve
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, EPOCHS+1), train_log, marker='o', label='train', alpha=0.8)
ax.plot(range(1, EPOCHS+1), val_log, marker='s', label='val', alpha=0.8)
ax.set_xlabel('epoch'); ax.set_ylabel('displacement MSE (nm²)')
ax.set_yscale('log'); ax.legend(); ax.grid(alpha=0.3)
ax.set_title('CellGNN training — multi-trajectory (8 train / 2 val)')
plt.tight_layout(); plt.show()

## 10. Roll out the trained model

Load the best-val checkpoint, initialize from a held-out trajectory's frame 0, predict next-frame displacements, integrate forward. Save the rollout for visualization or downstream feature extraction.

In [ ]:
model.eval()
ROLLOUT_FRAMES = 200

# Initialize from the first frame of a held-out validation trajectory
val_start = int(np.where(TRAJ_ID == N_TRAIN)[0][0])
valid0 = SPECIES_ID[val_start] >= 0
p0 = torch.tensor(POSITIONS[val_start][valid0], device=device).float()
s0 = torch.tensor(SPECIES_ID[val_start][valid0], device=device).long()
if p0.shape[0] > 2000:
    sel = torch.randperm(p0.shape[0])[:2000]
    p0 = p0[sel]; s0 = s0[sel]
n_p = p0.shape[0]
ex = torch.zeros((n_p, 4), device=device)

rollout = np.zeros((ROLLOUT_FRAMES, n_p, 3), dtype=np.float32)
rollout[0] = p0.cpu().numpy()
pos = p0.clone()
with torch.no_grad():
    for f in range(1, ROLLOUT_FRAMES):
        gg = build_dynamic_edges(pos, s0, r_cut_spatial=20.0)
        dp = model.predict_forces_equivariant(
            s0, ex, gg['edges'].to(device), gg['r'].to(device),
            gg['edge_type'].to(device), pos,
            thickness=gg['thickness'].to(device))
        pos = pos + dp
        rollout[f] = pos.cpu().numpy()

out_path = Path(REPO) / 'outputs/cell_gnn_rollout.npz'
np.savez_compressed(out_path,
                    positions=rollout, species_id=s0.cpu().numpy(),
                    t_ns=np.arange(ROLLOUT_FRAMES, dtype=np.float32))
print(f'rolled out {ROLLOUT_FRAMES} frames at 1 ns cadence (matches training)')
print(f'wrote {out_path}  ({out_path.stat().st_size/1e6:.1f} MB)')
print(f'mean radius:  start={np.linalg.norm(rollout[0], axis=-1).mean():.1f} nm  '
      f'end={np.linalg.norm(rollout[-1], axis=-1).mean():.1f} nm')

## 11. Compare rollout vs ground truth

Two diagnostics:
- **Mean radial extent** over time (rollout vs ground-truth frame 0..ROLLOUT_FRAMES if available)
- **Per-species mean position drift** to spot whether the model is collapsing or exploding

In [ ]:
rollout_r = np.linalg.norm(rollout, axis=-1).mean(axis=1)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(np.arange(ROLLOUT_FRAMES) * DT_US, rollout_r, label='rollout')
if T_FRAMES >= ROLLOUT_FRAMES:
    gt_r = np.linalg.norm(POSITIONS[:ROLLOUT_FRAMES], axis=-1)
    gt_mask = SPECIES_ID[:ROLLOUT_FRAMES] >= 0
    gt_r[~gt_mask] = np.nan
    axes[0].plot(np.arange(ROLLOUT_FRAMES) * DT_US,
                  np.nanmean(gt_r, axis=1), '--', label='ground truth')
axes[0].set_xlabel('t (μs)'); axes[0].set_ylabel('mean radius (nm)')
axes[0].set_title('mean radial extent'); axes[0].legend(); axes[0].grid(alpha=0.3)

# 2D projection of frames 0, mid, last
for k, idx in enumerate([0, ROLLOUT_FRAMES // 2, ROLLOUT_FRAMES - 1]):
    axes[1].scatter(rollout[idx, :, 0], rollout[idx, :, 1],
                     s=2, alpha=0.4, label=f'f{idx}')
axes[1].set_xlabel('x (nm)'); axes[1].set_ylabel('y (nm)')
axes[1].set_aspect('equal'); axes[1].set_title('xy-projection of rollout')
axes[1].legend()
plt.tight_layout(); plt.show()

## What we have

- Auto-discovery of trajectory data on drive
- Conversion to particle list — works for both LM HDF5 and our own .npz
- Heterogeneous edge graph built per frame, 7 edge types available
- CellGNN forward, equivariance verified
- Tiny next-frame-displacement training
- Rollout at 1 μs cadence saved to `outputs/cell_gnn_rollout.npz`

## Next steps

- **Reaction labels** — extract per-frame reaction events from LM data (write to `reaction_logits` target). Currently only forces are trained.
- **Spawn/destroy labels** — track births/deaths frame-to-frame and train the spawn head.
- **Particle ID tracking for real LM data** — current pair-builder relies on slot ordering; LM frames need a permutation match (e.g. greedy nearest-neighbour).
- **Push the rollout per-gene latent state** into `scripts/sparse_lnn_cascade_stacker.py` as a new feature block. Re-run LOO and compare to current baseline.